In [1]:
import pickle

import pandas as pd
import datetime as dt
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.feature_extraction import DictVectorizer

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet

from sklearn.metrics import root_mean_squared_error

In [2]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    # df['duration'] = (df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']).dt.total_seconds() / 60

    # df = df[ (df['duration'] >= 1) & (df['duration'] <= 60) ]

    # categorical = ['PULocationID', 'DOLocationID']
    # df[categorical] = df[categorical].astype(str)
    
    return df

In [13]:
df_train = read_dataframe('../data/yellow_tripdata_2023-01.parquet')

In [14]:
print(f"Total columns: {len(df_train.columns)}" )

Total columns: 19


In [15]:
df_train['duration'] = (df_train['tpep_dropoff_datetime'] - df_train['tpep_pickup_datetime']).dt.total_seconds() / 60

In [16]:
print(f"Standard deviation of the trips duration in January is: {float(df_train['duration'].std())}")
bf_duration_total = len(df_train['duration'])
print(f"The total number of entries is: {bf_duration_total}")

Standard deviation of the trips duration in January is: 42.59435124195457
The total number of entries is: 3066766


In [17]:
df_train = df_train[ (df_train['duration'] >= 1 ) & ( df_train['duration'] <= 60 ) ]

In [18]:
af_duration_total = len(df_train['duration'])
print(f"The total number of entries after filtering: {af_duration_total}")
print(f"Fraction of the records left after you dropped the outliers is: {af_duration_total / bf_duration_total}")

The total number of entries after filtering: 3009173
Fraction of the records left after you dropped the outliers is: 0.9812202822125979


In [59]:
categorical = ['PULocationID', 'DOLocationID']

df_train[categorical] = df_train[categorical].astype(int).astype(str)

dv = DictVectorizer()

train_dicts = df_train[categorical].to_dict(orient = 'records')
X_train = dv.fit_transform(train_dicts)

In [48]:
x_shape = X_train.shape[1]
print(f"dimensionality of this matrix (number of columns): {x_shape} ")

dimensionality of this matrix (number of columns): 515 


In [49]:
target = 'duration'
y_train = df_train[target].values

In [50]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_train)
rmse = root_mean_squared_error(y_train,y_pred)
print(rmse)

7.649261932456945


In [52]:
df_val = read_dataframe("../data/yellow_tripdata_2023-02.parquet")

In [53]:
df_val['duration'] = (df_val['tpep_dropoff_datetime'] - df_val['tpep_pickup_datetime']).dt.total_seconds() / 60

In [54]:
df_val = df_val[ (df_val['duration'] >= 1 ) & ( df_val['duration'] <= 60 ) ]

In [55]:
df_val[categorical] = df_val[categorical].astype(int).astype(str)

In [60]:
val_dicts = df_val[categorical].to_dict(orient = 'records')
X_val = dv.transform(val_dicts)

In [61]:
y_val = df_val[target].values

In [62]:
y_val_pred = lr_model.predict(X_val)
rmse = root_mean_squared_error(y_val,y_val_pred)
print(rmse)

7.811818758819308
